In [18]:
import pandas as pd
import numpy as np
from math import acos, degrees
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [19]:
FILENAME = 'DJI_0057.csv'
FILE_PATH = f'csv_files/{FILENAME}'
MAP_KEYPOINTS = {0:'nose', 1:'leye', 2:'reye', 3:'lear', 4:'rear', 5:'lshoulder', 6:'rshoulder', 7:'lelbow', 8:'relbow',
                 9:'lwrist', 10:'rwrist', 11:'lhip', 12:'rhip', 13:'lknee', 14:'rknee', 15:'lankle', 16:'rankle'}
SPEED_CALC_INTERVAL = 29
MAP_DIRECTION = {-1: 'UNKNOWN', 0: 'LEFT', 1: 'RIGHT', 2: 'UP', 3: 'DOWN'}


In [20]:
df = pd.read_csv(FILE_PATH, header=None)
print(df.shape)
for i in range(3,df.shape[1]):
    df[i] =  df[i].apply(lambda x: x.replace('[','').replace(']','')) 

(846, 37)


In [21]:
# only applicable when keypoint 0 is column '3'
# 0->3
# 1->5
# 2->7

In [22]:
def distance(ax,ay,bx,by):
    return np.sqrt((ax-bx)**2+(ay-by)**2)

def angle(ax,ay,bx,by,cx,cy):
    import math
    ang = degrees(math.atan2(cy-by, cx-bx) - math.atan2(ay-by, ax-bx))
    return ang + 360 if ang < 0 else ang


In [23]:
distance_features = []
for i in range(17):
    for j in range(i+1,17):
        # x, y
        ax, ay = df[i*2+3].astype(float),df[i*2+4].astype(float)
        bx, by = df[j*2+3].astype(float),df[j*2+4].astype(float)
        colname = MAP_KEYPOINTS[i] + '_' + MAP_KEYPOINTS[j]
        df[colname] = distance(ax,ay,bx,by)
        distance_features.append(colname)

In [24]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,rhip_lknee,rhip_rknee,rhip_lankle,rhip_rankle,lknee_rknee,lknee_lankle,lknee_rankle,rknee_lankle,rknee_rankle,lankle_rankle
0,False,1,-1.0,737.5023803710938,320.6878967285156,741.2019653320312,319.67144775390625,740.672607421875,322.5329895019531,738.1246948242188,...,69.487206,67.094423,124.947135,124.623740,15.394857,56.803790,60.205918,58.452770,57.630846,16.048188
1,False,0,-1.0,583.844482421875,308.04400634765625,580.197509765625,311.2861633300781,580.4197998046875,304.9407958984375,584.1820678710938,...,53.740951,48.909821,91.579467,89.143095,21.660621,39.929666,44.751245,45.412915,40.248367,19.638807
2,False,0,-1.0,579.7529907226562,308.1734313964844,576.074462890625,310.5860595703125,576.4519653320312,305.9104309082031,579.6644287109375,...,48.107642,43.475880,81.759793,79.045900,19.942441,35.510492,39.696724,40.991703,35.591829,18.261971
3,False,0,-1.0,573.4923706054688,307.8982238769531,569.5716552734375,310.6782531738281,570.0569458007812,305.23248291015625,573.339111328125,...,50.936545,46.437132,86.596467,84.143965,20.167181,37.627159,41.816421,42.629248,37.720668,18.029682
4,False,0,-1.0,569.0383911132812,306.30621337890625,565.4944458007812,306.4144592285156,565.327392578125,306.0070495605469,569.094970703125,...,43.357239,37.189921,69.473637,65.721808,20.779240,28.809589,34.960972,35.207831,28.535791,19.674849


In [25]:
angle_features = []
for i in range(17):
    for j in range(i+1,17):
        for k in range(j+1, 17):
            # x, y
            ax, ay = df[i*2+3].astype(float),df[i*2+4].astype(float)
            bx, by = df[j*2+3].astype(float),df[j*2+4].astype(float)
            cx, cy = df[k*2+3].astype(float),df[k*2+4].astype(float)
            colname = MAP_KEYPOINTS[i] + '_' + MAP_KEYPOINTS[j] + '_' + MAP_KEYPOINTS[k]
            angle_array = []
            for idx in range(len(ax)):
                angle_array.append(angle(ax[idx],ay[idx],bx[idx],by[idx],cx[idx],cy[idx]))
            df[colname] = angle_array
            angle_features.append(colname)

In [26]:
df.rename(columns={0:'red_marker', 1:'direction', 2:'speed'}, inplace=True)
df = df[['red_marker','direction','speed']+distance_features+angle_features]


In [27]:
groups = df.groupby(np.arange(len(df.index))//SPEED_CALC_INTERVAL)
rows = []
for (frameno, frame) in groups:
    red_marker_ = np.any(frame['red_marker'])
    direction_ = max(frame['direction'])
    speed_ = max(frame['speed'])
    mean_ = list(frame.iloc[:,3:].mean())
    rows.append([red_marker_, direction_, speed_] + mean_)
    # print(mean_)
    # break
    # print(red_marker_)

In [28]:
final_df = pd.DataFrame(rows, columns=['red_marker','direction','speed']+distance_features+angle_features)
final_df['direction'] = final_df['direction'].map(lambda x: MAP_DIRECTION[x])

final_df['time'] = np.arange(0,final_df.shape[0]*0.5,0.5)
final_df['time'] = final_df['time'].map(lambda x: f'{x//60}:{x%60}')
final_df = final_df[['time','red_marker','direction','speed']+distance_features+angle_features]
final_df['speed_pct_change'] = final_df['speed'].pct_change()

In [29]:
for i in angle_features:
    final_df[i] = np.unwrap(final_df[i], period=360)


In [30]:
distance_features_pct = [i + '_pct' for i in distance_features]
for i,j in zip(distance_features_pct,distance_features):
    final_df[i] = final_df[j].pct_change()


angle_features_pct = [i + '_pct' for i in angle_features]
for i,j in zip(angle_features_pct,angle_features):
    final_df[i] = final_df[j].pct_change()

In [31]:
def find_top_correlation(row):
    idx_max = row.argmax()
    value_max = row[idx_max]
    colname_max = row.index[idx_max]

    idx_min = row.argmin()
    value_min = row[idx_min]
    colname_min = row.index[idx_min]
    return f'{colname_max}:{value_max}, {colname_min}:{value_min}'

In [32]:
# final_df['correlation'] = final_df[distance_features_pct+angle_features_pct].apply(lambda row: find_top_correlation(row), axis=1)
final_df['distance_correlation'] = final_df[distance_features_pct].apply(lambda row: find_top_correlation(row), axis=1)
final_df['angle_correlation'] = final_df[angle_features_pct].apply(lambda row: find_top_correlation(row), axis=1)

C:\Users\Admin\AppData\Local\Temp\ipykernel_28664\202334822.py:2: FutureWarning: The behavior of Series.argmax/argmin with skipna=False and NAs, or with all-NAs is deprecated. In a future version this will raise ValueError.
  idx_max = row.argmax()
C:\Users\Admin\AppData\Local\Temp\ipykernel_28664\202334822.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  value_max = row[idx_max]
C:\Users\Admin\AppData\Local\Temp\ipykernel_28664\202334822.py:6: FutureWarning: The behavior of Series.argmax/argmin with skipna=False and NAs, or with all-NAs is deprecated. In a future version this will raise ValueError.
  idx_min = row.argmin()
C:\Users\Admin\AppData\Local\Temp\ipykernel_28664\202334822.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys w

In [33]:
final_df['angle_correlation']

0     rknee_lankle_rankle_pct:nan, rknee_lankle_rank...
1     rear_lwrist_lhip_pct:1.316187746489327, rear_l...
2     leye_lwrist_rknee_pct:46.57458504406501, rear_...
3     leye_rwrist_lhip_pct:56.218452738702, nose_rey...
4     rear_lwrist_lhip_pct:3.1179055760893633, lear_...
5     rshoulder_relbow_rknee_pct:3.4681407356991194,...
6     rshoulder_relbow_rhip_pct:37.62376585805342, n...
7     nose_lshoulder_relbow_pct:10.165620114216896, ...
8     leye_lwrist_rknee_pct:30.06975096595031, rshou...
9     nose_leye_lhip_pct:3.234333521426227, reye_lwr...
10    leye_rwrist_lhip_pct:18.498976485475044, nose_...
11    nose_lwrist_rankle_pct:14.564005592981722, rel...
12    relbow_rwrist_lhip_pct:1.0893413193805959, nos...
13    nose_reye_lshoulder_pct:2.2147614311540016, no...
14    nose_lear_rear_pct:2.162737394243123, nose_ley...
15    lshoulder_lwrist_rhip_pct:0.9004056880284463, ...
16    nose_reye_lshoulder_pct:2.494652365734246, lel...
17    lelbow_lwrist_rhip_pct:0.9983041863909334,

In [34]:
final_df[['time','red_marker','direction','speed','speed_pct_change','distance_correlation','angle_correlation']].to_csv(f'cleaned_csv_files/{FILENAME}')

In [ ]:
# print(final_df.shape)
# final_df.to_csv(f'cleaned_csv_files/{FILENAME}')


(30, 820)
